# 02 — Cointegration Screening

Runs Engle-Granger two-step pair selection over the 2018–2020 training window,
applies BH FDR correction within sectors, then half-life and Hurst filters.

**Window:** 2018-01-01 to 2020-12-31 (~756 trading days)
**Min history:** 504 days (~2 years) — tickers shorter than this are excluded per pair

In [ ]:
import sys
from pathlib import Path

# Add project root to path so src.* imports work from the notebooks/ subdirectory
sys.path.insert(0, str(Path('..').resolve()))

import logging
import warnings

import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import numpy as np
import pandas as pd
import seaborn as sns

from src.data import ASX_UNIVERSE, fetch_history, get_close_panel
from src.cointegration import screen_pairs
from src.filters import apply_all_filters, filter_half_life, filter_hurst, filter_significance

logging.basicConfig(
    level=logging.INFO,
    format='%(asctime)s [%(levelname)s] %(message)s',
)
warnings.filterwarnings('ignore', category=FutureWarning)
sns.set_theme(style='whitegrid', font_scale=0.9)

FIGURES_DIR = Path('../reports/figures')
FIGURES_DIR.mkdir(parents=True, exist_ok=True)

In [ ]:
# Load cached prices
tickers = [t for tickers in ASX_UNIVERSE.values() for t in tickers]
df = fetch_history(tickers, start='2010-01-01', cache_dir='../data/cache')
prices = get_close_panel(df)  # adj_close, wide panel: rows=date, cols=ticker

print(f'Panel shape:  {prices.shape}')
print(f'Date range:   {prices.index.min().date()} to {prices.index.max().date()}')
print(f'Tickers:      {prices.shape[1]}')

## 1. Screen pairs: 2018–2020 window

In [ ]:
WINDOW_START = '2018-01-01'
WINDOW_END   = '2020-12-31'

all_pairs = screen_pairs(
    prices=prices,
    sectors=ASX_UNIVERSE,
    window_start=WINDOW_START,
    window_end=WINDOW_END,
    fdr=0.10,
    min_history_days=504,
)

print(f'Total pairs tested:    {len(all_pairs)}')
print(f'Sectors with ≥2 tickers that produced valid pairs: {all_pairs["sector"].nunique()}')

## 2. Single-test sector warning

Sectors with exactly one surviving pair after the min-history guard are flagged.
BH correction does nothing meaningful on a single test — the raw ADF p < 0.05 threshold
is used instead. These pairs are **not** multiple-testing-corrected and must be called
out in the Phase 5 writeup.

In [ ]:
single = all_pairs[all_pairs['single_test_sector']]

if len(single) > 0:
    print('WARNING: single-test sectors (BH correction NOT applied):')
    for _, row in single.iterrows():
        status = 'PASS (p<0.05)' if row['bh_pass'] else f'FAIL (p={row["adf_pvalue"]:.4f})'
        print(f'  {row["sector"]}: {row["ticker_a"]}/{row["ticker_b"]}  '
              f'ADF p={row["adf_pvalue"]:.4f}  {status}')
else:
    print('No single-test sectors in this window.')

## 3. Filter funnel

In [ ]:
after_sig   = filter_significance(all_pairs)
after_hl    = filter_half_life(after_sig)
after_hurst = filter_hurst(after_hl)

funnel = [
    ('All pairs tested',                      len(all_pairs)),
    ('After significance (BH 10% FDR)',        len(after_sig)),
    ('After half-life filter (5–60 days)',      len(after_hl)),
    ('After Hurst filter (H < 0.5)',            len(after_hurst)),
]

for label, count in funnel:
    print(f'  {count:3d}  {label}')

In [ ]:
# Funnel bar chart
labels = [f[0] for f in funnel]
counts = [f[1] for f in funnel]

fig, ax = plt.subplots(figsize=(9, 4))
bars = ax.barh(labels[::-1], counts[::-1], color='steelblue', edgecolor='white')
for bar, count in zip(bars, counts[::-1]):
    ax.text(bar.get_width() + 0.3, bar.get_y() + bar.get_height() / 2,
            str(count), va='center', fontsize=10)
ax.set_xlabel('Number of pairs')
ax.set_title('Pair selection funnel — 2018-01-01 to 2020-12-31')
ax.set_xlim(0, max(counts) * 1.15)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'pair_funnel_2018_2020.png', dpi=150, bbox_inches='tight')
plt.show()

## 4. Top 10 pairs by ADF p-value — spread plots

Spreads recomputed from the estimated beta and alpha. Mean and ±2σ bands shown.
These are the pairs most likely to survive all filters.

In [ ]:
top10 = all_pairs.sort_values('adf_pvalue').head(10).reset_index(drop=True)

window_prices = prices.loc[WINDOW_START:WINDOW_END]

fig, axes = plt.subplots(5, 2, figsize=(16, 22))
axes = axes.flatten()

for i, row in top10.iterrows():
    ax = axes[i]
    a, b = row['ticker_a'], row['ticker_b']

    log_a = np.log(window_prices[a].dropna())
    log_b = np.log(window_prices[b].dropna())
    shared = log_a.index.intersection(log_b.index)
    spread = log_a.loc[shared] - row['beta'] * log_b.loc[shared] - row['alpha']

    mean = spread.mean()
    std  = spread.std()

    ax.plot(spread.index, spread.values, linewidth=0.8, color='steelblue', alpha=0.85)
    ax.axhline(mean,          color='black', linewidth=1.0,  linestyle='-',  label='mean')
    ax.axhline(mean + 2 * std, color='crimson', linewidth=0.8, linestyle='--', label='+2σ')
    ax.axhline(mean - 2 * std, color='crimson', linewidth=0.8, linestyle='--', label='-2σ')

    hl  = row['half_life_days']
    h   = row['hurst']
    hl_str = f'{hl:.1f}d' if not np.isnan(hl) else 'HL=NaN'
    h_str  = f'H={h:.2f}' if not np.isnan(h)  else 'H=NaN'

    ax.set_title(
        f'{a} / {b}\np={row["adf_pvalue"]:.4f}  {hl_str}  {h_str}',
        fontsize=9,
    )
    ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y-%m'))
    ax.tick_params(axis='x', rotation=45, labelsize=7)
    if i == 0:
        ax.legend(fontsize=7, loc='upper right')

plt.suptitle('Top 10 pairs by ADF p-value (2018-2020 window)', y=1.01, fontsize=11)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'top10_spreads_2018_2020.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Economic sanity check

These pairs should appear in the screened output for any reasonable training window.
If they don't, something is wrong upstream (data gaps, wrong price column, log-price not taken).

In [ ]:
EXPECTED = [
    ('CBA.AX', 'WBC.AX'),
    ('ANZ.AX', 'NAB.AX'),
    ('CBA.AX', 'ANZ.AX'),
    ('BHP.AX', 'RIO.AX'),
    ('GMG.AX', 'SGP.AX'),
    ('IAG.AX', 'SUN.AX'),
]

print('Economic sanity check (looking in all_pairs, pre-filter):')
found_count = 0
for a, b in EXPECTED:
    match = all_pairs[
        ((all_pairs['ticker_a'] == a) & (all_pairs['ticker_b'] == b)) |
        ((all_pairs['ticker_a'] == b) & (all_pairs['ticker_b'] == a))
    ]
    if len(match) > 0:
        r = match.iloc[0]
        bh  = 'BH-PASS' if r['bh_pass'] else 'BH-FAIL'
        hl  = f"{r['half_life_days']:.1f}d" if not np.isnan(r['half_life_days']) else 'HL=NaN'
        h   = f"H={r['hurst']:.2f}" if not np.isnan(r['hurst']) else 'H=NaN'
        print(f'  FOUND  {a}/{b}  p={r["adf_pvalue"]:.4f}  {bh}  {hl}  {h}')
        found_count += 1
    else:
        print(f'  MISSING {a}/{b}  (excluded by min-history guard or not in universe)')

print(f'\n{found_count}/{len(EXPECTED)} expected pairs found in screened output.')
if found_count < 3:
    print('WARNING: fewer than 3 expected pairs found — check data quality or log-price step.')

## 6. Final surviving pairs

In [ ]:
survivors = apply_all_filters(all_pairs)

print(f'Survivors after all filters: {len(survivors)}')
display_cols = ['ticker_a', 'ticker_b', 'sector', 'adf_pvalue',
                'half_life_days', 'hurst', 'bh_pass', 'single_test_sector']
print(survivors[display_cols].to_string(index=False, float_format='{:.4f}'.format))